# Random Forest Regression

## Import Libraries

In [ ]:
# Standard library imports for data handling
import pandas as pd
import numpy as np
import joblib

# Import scikit-learn tools for model selection and evaluation
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

# Import categorical encoders for handling make and model features
from category_encoders import TargetEncoder

## Load Final Dataset

In [ ]:
# Define the path for the processed combined dataset
DATA_PATH = '../data/processed/combined_final_dataset.csv'

# Read the CSV file into a pandas dataframe
df = pd.read_csv(DATA_PATH)

# Output the dimensions of the loaded dataset for verification
print(f"Loaded prepared dataset: {df.shape[0]} rows x {df.shape[1]} columns")

# Display the first few records of the dataframe
df.head()

Loaded prepared dataset: 691418 rows x 11 columns


,country,make,model,year,age,mileage,transmission,fuelType,mpg,engineSize,price
0,uk,vauxhall,corsa,2018,2,9876.0,manual,petrol,46.13158,1.4,10124.340
1,uk,vauxhall,corsa,2019,1,2500.0,manual,petrol,45.21561,1.4,15401.580
2,uk,vauxhall,corsa,2017,3,9625.0,automatic,petrol,39.88633,1.4,12553.668
3,uk,vauxhall,corsa,2016,4,25796.0,manual,petrol,46.13158,1.4,10914.000
4,uk,vauxhall,corsa,2019,1,3887.0,manual,petrol,36.22245,1.4,12840.000


## Data Preprocessing

### Handle year column, and EV mpg and engineSize

In [ ]:
# Initialize random seed and test split ratio for reproducibility
RANDOM_SEED = 42
TEST_SIZE = 0.2

# Identify and drop rows with missing or invalid target values
df = df.dropna(subset=['price'])
df = df[df['price'] > 0]

# Output the new dataset shape after target filtering
print(f"Dataset after filtering: {df.shape}")

# Drop the year column to prevent multicollinearity with the age feature
if 'year' in df.columns:
    df = df.drop(columns=['year'])
    print("Dropped 'year' column")

# Create a binary is_ev flag to identify electric vehicles for special handling
df['is_ev'] = (df['fuelType'] == 'electric').astype(int)

# Output the frequency and percentage of EV records in the dataset
print(f"EV records: {df['is_ev'].sum()} / {len(df)} ({df['is_ev'].mean()*100:.2f}%)")

# Calculate global medians for MPG and engine size from non-electric vehicles
non_ev_mpg_median = df.loc[df['is_ev'] == 0, 'mpg'].median()
non_ev_engine_median = df.loc[df['is_ev'] == 0, 'engineSize'].median()

# Impute electric vehicle MPG values with the calculated non-EV median
df.loc[df['is_ev'] == 1, 'mpg'] = non_ev_mpg_median

# Impute electric vehicle engine size values with the calculated non-EV median
df.loc[df['is_ev'] == 1, 'engineSize'] = non_ev_engine_median

# Output the imputation values used for the EV records
print(f"Imputed EV mpg with {non_ev_mpg_median:.1f}, engineSize with {non_ev_engine_median:.1f}")

Dataset after filtering: (691418, 11)
Dropped 'year' column
EV records: 13855 / 691418 (2.00%)
Imputed EV mpg with 25.5, engineSize with 2.4


### Set train/test split

In [ ]:
# Define the target variable for the regression problem
TARGET = 'price'

# Separate the feature matrix (X) from the target vector (y)
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Perform a stratified split to ensure consistent EV representation across sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED,
    stratify=X['is_ev']
)

# Output the number of samples in the training and testing partitions
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:      {X_test.shape[0]} samples")

# Calculate sample weights to prioritize UK market accuracy in the model
train_weights = np.where(X_train['country'] == 'UK', 4.0, 0.5)

# Output confirmation of sample weight assignment
print("\nSample weights computed (UK=4.0, US=0.5).")

Training set: 553134 samples
Test set:     138284 samples

Sample weights computed (UK=4.0, US=0.5).


### Encoding Categorical Features

#### One-Hot Encode Low Cardinality Features

In [ ]:
# Define categorical features with low cardinality for one-hot encoding
low_cat_features = ['country', 'transmission', 'fuelType']

# Convert categorical variables into dummy features for the training set
X_train = pd.get_dummies(X_train, columns=low_cat_features, drop_first=True)

# Convert categorical variables into dummy features for the test set
X_test = pd.get_dummies(X_test, columns=low_cat_features, drop_first=True)

# Align the train and test dataframes to ensure matching column structures
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# Extract and store the final feature names for the deployment bundle
features = list(X_train.columns)

#### Target Encode High Cardinality Features

In [ ]:
# Identify high-cardinality features for target encoding
high_cat_features = ['make', 'model']

# Initialize the ColumnTransformer to handle target encoding for specified features
preprocessor = ColumnTransformer(
    transformers=[
        ('target_enc', TargetEncoder(smoothing=10.0), high_cat_features),
    ],
    remainder='passthrough'
)

# Construct the full modeling pipeline including preprocessing and scaling
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('regressor', RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1))
])

## Hyperparameter Tuning

In [ ]:
# Define the search space for Random Forest hyperparameter optimization
param_distributions = {
    'regressor__n_estimators': [100, 200, 300],          
    'regressor__max_depth': [10, 15, 20, None],          
    'regressor__max_features': ['sqrt', 'log2', 1.0]
}

# Initialize RandomizedSearchCV to find the optimal model configuration
model_random = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=20, 
    cv=3,      
    scoring='neg_mean_absolute_error',
    verbose=2, 
    random_state=RANDOM_SEED,
    n_jobs=-1  
)

# Output progress message before starting the search process
print("Starting Hyperparameter Tuning... This may take a few minutes.")

# Execute the search using the calculated sample weights for training
model_random.fit(X_train, y_train, regressor__sample_weight=train_weights)

# Extract the best estimator found during the search process
best_pipeline = model_random.best_estimator_

# Output the best hyperparameter combination to the console
print("\n=== Best Hyperparameters Found ===")
print(model_random.best_params_)

Starting Hyperparameter Tuning... This may take a few minutes.
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END regressor__max_depth=None, regressor__max_features=log2, regressor__n_estimators=100; total time= 2.7min
[CV] END regressor__max_depth=15, regressor__max_features=log2, regressor__n_estimators=200; total time= 2.8min
[CV] END regressor__max_depth=15, regressor__max_features=log2, regressor__n_estimators=200; total time= 2.8min
[CV] END regressor__max_depth=15, regressor__max_features=log2, regressor__n_estimators=200; total time= 2.8min
[CV] END regressor__max_depth=None, regressor__max_features=log2, regressor__n_estimators=100; total time= 2.6min
[CV] END regressor__max_depth=None, regressor__max_features=log2, regressor__n_estimators=100; total time= 2.6min
[CV] END regressor__max_depth=None, regressor__max_features=log2, regressor__n_estimators=200; total time= 5.3min
[CV] END regressor__max_depth=15, regressor__max_features=1.0, regressor__n_estimato

/Users/karimamr/miniconda3/envs/usedCars/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END regressor__max_depth=15, regressor__max_features=sqrt, regressor__n_estimators=100; total time= 1.6min
[CV] END regressor__max_depth=15, regressor__max_features=sqrt, regressor__n_estimators=100; total time= 1.6min
[CV] END regressor__max_depth=10, regressor__max_features=sqrt, regressor__n_estimators=100; total time= 1.3min
[CV] END regressor__max_depth=10, regressor__max_features=sqrt, regressor__n_estimators=100; total time= 1.4min
[CV] END regressor__max_depth=10, regressor__max_features=sqrt, regressor__n_estimators=100; total time= 1.3min
[CV] END regressor__max_depth=10, regressor__max_features=1.0, regressor__n_estimators=300; total time=10.6min
[CV] END regressor__max_depth=10, regressor__max_features=1.0, regressor__n_estimators=300; total time=10.6min
[CV] END regressor__max_depth=10, regressor__max_features=log2, regressor__n_estimators=200; total time= 2.3min
[CV] END regressor__max_depth=10, regressor__max_features=1.0, regressor__n_estimators=300; total time=10.

/Users/karimamr/miniconda3/envs/usedCars/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/karimamr/miniconda3/envs/usedCars/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/karimamr/miniconda3/envs/usedCars/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/karimamr/min

[CV] END regressor__max_depth=10, regressor__max_features=sqrt, regressor__n_estimators=200; total time= 1.4min
[CV] END regressor__max_depth=15, regressor__max_features=sqrt, regressor__n_estimators=300; total time= 2.8min
[CV] END regressor__max_depth=15, regressor__max_features=sqrt, regressor__n_estimators=300; total time= 2.8min
[CV] END regressor__max_depth=15, regressor__max_features=sqrt, regressor__n_estimators=300; total time= 2.6min

=== Best Hyperparameters Found ===
{'regressor__n_estimators': 200, 'regressor__max_features': 'sqrt', 'regressor__max_depth': 20}


### Model Assessment

#### Check for overfitting

In [ ]:
# Generate predictions on the training data to assess memorization
y_train_pred = best_pipeline.predict(X_train)

# Calculate the R-squared score for the training set
train_r2 = r2_score(y_train, y_train_pred)

# Generate predictions on the unseen test data to assess generalization
y_test_pred = best_pipeline.predict(X_test)

# Calculate the R-squared score for the testing set
test_r2 = r2_score(y_test, y_test_pred)

# Output the overfitting audit results for comparison
print("Overfitting Checker")
print(f"Training R²: {train_r2:.4f}")
print(f"Testing R²:  {test_r2:.4f}")
print(f"Difference:  {(train_r2 - test_r2):.4f}")

=== Overfitting Diagnostic ===
Training R²: 0.9727
Testing R²:  0.9520
Difference:  0.0207


#### R2, MAE, MAPE and RMSE

In [ ]:
# Generate final predictions on the raw test set using the tuned pipeline
y_pred = best_pipeline.predict(X_test)

# Calculate Mean Absolute Error (MAE) in currency units
mae = mean_absolute_error(y_test, y_pred)

# Calculate Mean Absolute Percentage Error (MAPE) for error scale context
mape = mean_absolute_percentage_error(y_test, y_pred) * 100

# Calculate Root Mean Squared Error (RMSE) to penalize larger errors
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# Calculate the final R-squared score for the predictive model
r2 = r2_score(y_test, y_pred)

# Output the final performance summary to the console
print("\n=== Tuned Pipeline Performance ===")
print(f"MAE:  £{mae:.2f}")
print(f"MAPE: {mape:.2f}%")
print(f"RMSE: £{rmse:.2f}")
print(f"R²:   {r2:.4f}")


=== Tuned Pipeline Performance ===
MAE:  £1877.06
MAPE: 8.50%
RMSE: £2748.08
R²:   0.9520

Success! Deployment bundle saved to: ../models/model.pkl


#### Model Export

In [ ]:
# Initialize the deployment bundle including the pipeline and metadata
deployment_bundle = {
    "pipeline": best_pipeline,
    "non_ev_mpg_median": non_ev_mpg_median,
    "non_ev_engine_median": non_ev_engine_median,
    "feature_columns": list(X_train.columns) 
}

# Define the final export path for the model pickle file
EXPORT_PATH = '../models/model.pkl'

# Save the deployment bundle to the specified directory using joblib
joblib.dump(deployment_bundle, EXPORT_PATH)

# Output confirmation of the successful model export
print(f"\nSuccess! Deployment bundle saved to: {EXPORT_PATH}")